Luetaan jokainen seitsemästä CSV-tiedostosta Volumesta ja tallennetaan se omaksi Delta-taulukseen (bronze_<nimi>). Bronze-tasolla dataa ei muokata sisällöllisesti. Tyypit arvataan automaattisesti (inferSchema), ja jokaiselle riville lisätään kaksi metadatasaraketta: latausaika (_ingested_at) ja lähdetiedosto (_source_file). Silmukka tekee saman jokaiselle taululle, joten koodia ei tarvitse toistaa seitsemää kertaa. overwrite korvaa taulun uudelleenajossa.

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, count, when, isnan, expr

raw_path = "/Volumes/workspace/default/football_database/football_database"

tables = ["appearances", "games", "leagues", "players", "shots", "teams", "teamstats"]

# False: ohita taulut, jotka on jo ladattu. True: lataa kaikki uudelleen CSV:istä.
FORCE_RELOAD = False

for t in tables:
    table_name = f"bronze_{t}"

    if spark.catalog.tableExists(table_name) and not FORCE_RELOAD:
        print(f"{table_name}: olemassa, ohitettu")
        continue

    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .csv(f"{raw_path}/{t}.csv"))

    df = (df
          .withColumn("_ingested_at", current_timestamp())
          .withColumn("_source_file", lit(f"{t}.csv")))

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table_name))

    print(f"{table_name}: ladattu")

Tarkistetaan, että jokainen taulu latautui ja että rivi- ja sarakemäärät ovat järkeviä. Metadatasarakkeet on laskettu mukaan sarakemääriin.

In [0]:
for t in tables:
    df = spark.table(f"bronze_{t}")
    print(f"{t}: {df.count()} riviä, {len(df.columns)} saraketta")

In [0]:
Listataan skeeman workspace.default taulut. Bronze-taulut näkyvät muodossa bronze_*. Tämä pitää ajaa omassa solussaan, koska %sql koskee koko solua.

In [0]:
%sql
SHOW TABLES IN workspace.default

Funktio käy läpi taulun jokaisen sarakkeen (metadatasarakkeita lukuun ottamatta) ja tulostaa sen tietotyypin, NULL-arvojen määrän ja osuuden sekä erilaisten arvojen lukumäärän. Siitä näkee, mitkä sarakkeet ovat luokittelevia ja mitkä jatkuvia. Tyyppi paljastaa myös ongelmat: numeeriselta kuulostava sarake sisältää todennäköisesti ei-numeerisia arvoja. Funktio ajetaan kaikille seitsemälle taululle.

In [0]:
tables = ["appearances", "games", "leagues", "players", "shots", "teams", "teamstats"]

def profile_table(table_name):
    df = spark.table(f"bronze_{table_name}")
    total_rows = df.count()
    print(f"\n=== {table_name} ({total_rows} riviä) ===")
    
    for c, dtype in df.dtypes:
        if c.startswith("_"):  # ohita metadata-sarakkeet
            continue
        null_count = df.filter(col(c).isNull()).count()
        distinct_count = df.select(c).distinct().count()
        null_pct = round(100 * null_count / total_rows, 1) if total_rows > 0 else 0
        print(f"  {c:<30} {dtype:<15} nullit: {null_count} ({null_pct}%)  eri arvoja: {distinct_count}")

for t in tables:
    profile_table(t)

Profiloinnin mukaan games-taulun 21 vedonlyötnikerroinsaraketta ovat tekstiä. Tässä etsitään, mitä arvoja niissä on, joita ei voi muuttaa luvuksi. try_cast(... AS DOUBLE) palauttaa NULLin, jos muunnos ei onnistu, joten suodatin löytää juuri ne rivit, joiden arvo ei ole kelvollinen luku. Tulokset kootaan sarakkeittain yhteen taulukkoon, jossa on sarake, arvo ja lukumäärä. Tulos: ainoa ei-numeerinen arvo on NA. Silver-tasolla se muutetaan NULLiksi tyyppimuunnoksen yhteydessä.

In [0]:
odds_cols = ["B365H","B365D","B365A","BWH","BWD","BWA","IWH","IWD","IWA",
             "PSH","PSD","PSA","WHH","WHD","WHA","VCH","VCD","VCA",
             "PSCH","PSCD","PSCA"]

games = spark.table("bronze_games")
results = None

for c in odds_cols:
    r = (games
         .filter(col(c).isNotNull() & expr(f"try_cast(`{c}` AS DOUBLE)").isNull())
         .groupBy(col(c).alias("arvo"))
         .count()
         .withColumn("sarake", lit(c)))
    results = r if results is None else results.unionByName(r)

results.select("sarake", "arvo", "count").orderBy("sarake", "arvo").show(200, truncate=False)

Sama tarkistus kahdelle muulle tekstinä tulleelle sarakkeelle. Kysely listaa kaikki arvot, joita ei voi muuttaa kokonaisluvuksi. Tulos: molemmissa ainoa tällainen arvo on NA. yellowCards-sarakkeessa se esiintyy yhdellä rivillä, assisterID-sarakkeessa 84 344 rivillä.

In [0]:
spark.sql("""
    SELECT yellowCards, COUNT(*) AS n
    FROM bronze_teamstats
    WHERE TRY_CAST(yellowCards AS INT) IS NULL
    GROUP BY yellowCards
""").show(50, truncate=False)

spark.sql("""
    SELECT assisterID, COUNT(*) AS n
    FROM bronze_shots
    WHERE TRY_CAST(assisterID AS INT) IS NULL
    GROUP BY assisterID
""").show(50, truncate=False)

Haetaan se yksi rivi, jonka keltaisten korttien määrä on NA. Kyseessä on ottelu 4888 (maaliskuu 2015, joukkue 95), jossa joukkue sai myös punaisen kortin. Arvo jätetään Silver-tasolla NULLiksi.

In [0]:
spark.sql("""
    SELECT gameID, teamID, date, location, goals, redCards, yellowCards
    FROM bronze_teamstats
    WHERE yellowCards = 'NA'
""").show(truncate=False)

Lasketaan, kuinka monella pelaajarivillä on kukin yhdistelmä keltaista (yellowCard) ja punaista (redCard) korttia. Yhdistelmä keltainen + punainen (1, 1) on harvinainen (110 riviä), joten kaikkia toisia keltaisia ei ole kirjattu näin. Osa on ilmeisesti kirjattu pelkkänä punaisena.

In [0]:
spark.sql("""
    SELECT yellowCard, redCard, COUNT(*) AS n
    FROM bronze_appearances
    GROUP BY yellowCard, redCard
    ORDER BY yellowCard, redCard
""").show()

Verrataan ottelukohtaisia keltaisten ja punaisten korttien summia kahdesta lähteestä: joukkuetaulusta (ts) ja pelaajataulusta (ap). Ottelu 4888 jätetään pois, koska sen keltaisten määrä puuttuu. Tulos kertoo, kuinka monessa ottelussa ero (joukkuetaulu miinus pelaajataulu) on 0, 1, 2 ja niin edelleen. Tulos: 92 % otteluista täsmää, punaiset kortit täsmäävät lähes aina ja keltaisissa on eroa noin 8 %:ssa otteluista. Syytä ei ole vielä varmistettu, ja se testataan seuraavassa solussa.

In [0]:
spark.sql("""
    WITH ts AS (
        SELECT gameID,
               SUM(try_cast(yellowCards AS INT)) AS ts_yellow,
               SUM(redCards) AS ts_red
        FROM bronze_teamstats
        WHERE gameID <> 4888
        GROUP BY gameID
    ),
    ap AS (
        SELECT gameID,
               SUM(yellowCard) AS ap_yellow,
               SUM(redCard) AS ap_red,
               SUM(CASE WHEN yellowCard = 1 AND redCard = 1 THEN 1 ELSE 0 END) AS both_cards
        FROM bronze_appearances
        GROUP BY gameID
    )
    SELECT ts.ts_yellow - ap.ap_yellow AS ero_keltaiset,
           ts.ts_red - ap.ap_red AS ero_punaiset,
           COUNT(*) AS otteluita
    FROM ts JOIN ap USING (gameID)
    GROUP BY 1, 2
    ORDER BY otteluita DESC
""").show(20)

Hypoteesi on, että joukkuetaulu laskee toisen keltaisen kahdeksi keltaiseksi, mutta pelaajataulu kirjaa sen pelkkänä punaisena. Jaetaan ottelut ryhmiin sen mukaan, onko niissä pelaaja, jolla on vain punainen kortti (has_red_only), ja pelaaja, jolla on sekä keltainen että punainen (has_yellow_and_red). Jokaiselle ryhmälle lasketaan keltaisten korttien keskimääräinen ero ja ottelut, joissa ero on tasan 2 tai 1. Jos hypoteesi pitää, ero 2 keskittyy otteluihin, joissa on pelkkä punainen, ja ero 1 otteluihin, joissa on molemmat kortit.

In [0]:
spark.sql("""
    WITH ts AS (
        SELECT gameID, SUM(try_cast(yellowCards AS INT)) AS ts_yellow
        FROM bronze_teamstats
        WHERE gameID <> 4888
        GROUP BY gameID
    ),
    ap AS (
        SELECT gameID,
               SUM(yellowCard) AS ap_yellow,
               SUM(CASE WHEN yellowCard = 0 AND redCard = 1 THEN 1 ELSE 0 END) AS red_only,
               SUM(CASE WHEN yellowCard = 1 AND redCard = 1 THEN 1 ELSE 0 END) AS yellow_and_red
        FROM bronze_appearances
        GROUP BY gameID
    )
    SELECT (ap.red_only > 0) AS has_red_only,
           (ap.yellow_and_red > 0) AS has_yellow_and_red,
           COUNT(*) AS games,
           ROUND(AVG(ts.ts_yellow - ap.ap_yellow), 2) AS avg_diff,
           SUM(CASE WHEN ts.ts_yellow - ap.ap_yellow = 2 THEN 1 ELSE 0 END) AS diff_2,
           SUM(CASE WHEN ts.ts_yellow - ap.ap_yellow = 1 THEN 1 ELSE 0 END) AS diff_1
    FROM ts JOIN ap USING (gameID)
    GROUP BY 1, 2
    ORDER BY 1, 2
""").show()

Lasketaan laukausten tilanteittain (situation), kuinka suurella osalla ei ole syöttäjää (assisterID = 'NA'). Rangaistuspotkuissa ja suorissa vapaapotkuissa osuus on 100 %, koska syöttäjää ei voi olla, ja muissa tilanteissa 20–30 %. NA tarkoittaa siis tässä sarakkeessa "ei sovellu", ei puuttuvaa tietoa. Silver-tasolla assisterID muutetaan kokonaisluvuksi ja lisätään erillinen hasAssist-sarake.

In [0]:
spark.sql("""
    SELECT situation,
           COUNT(*) AS shots,
           SUM(CASE WHEN assisterID = 'NA' THEN 1 ELSE 0 END) AS no_assister,
           ROUND(100 * AVG(CASE WHEN assisterID = 'NA' THEN 1.0 ELSE 0.0 END), 1) AS percentage
    FROM bronze_shots
    GROUP BY situation
    ORDER BY shots DESC
""").show(truncate=False)

substituteIn ja substituteOut: ensimmäinen tarkastelu

Profiloinnissa kumpikin sarake sai 77 560 eri arvoa, mikä ei sovi vaihtominuuttiin (mahdollisia arvoja olisi noin 90). Ensimmäinen kysely näyttää 20 riviä appearances-taulusta, jotta sarakkeita voi verrata positioon (position) ja pelattuun aikaan (time). Toinen kysely laskee yleisimmät substituteIn-arvot.

Tulos: 278 954 rivillä substituteIn on 0, ja jokainen muu arvo esiintyy vain kerran. Sarakkeet ovat siis tunnisteita, eivät minuutteja. Nollasta poikkeava substituteIn on pelaajilla, jotka vaihdettiin pois kesken ottelun, ja nollasta poikkeava substituteOut vaihtopelaajilla (position = Sub). Aloittajalla time on minuutti, jolloin hän lähti kentältä. Vaihtopelaajalla se on pelattu aika, joten tulominuutti on 90 − time. Esimerkiksi ottelussa 81 lähdöt minuuteilla 82, 61 ja 69 vastaavat vaihtopelaajien peliaikoja 8, 29 ja 21.

In [0]:
spark.table("bronze_appearances") \
    .select("gameID", "playerID", "position", "time", "substituteIn", "substituteOut") \
    .show(30, truncate=False)

spark.table("bronze_appearances") \
    .groupBy("substituteIn").count() \
    .orderBy("count", ascending=False).show(10)

Löytyykö vaihdoille tunnisteparit?

Testi tarkistaa, onko pois vaihdetun pelaajan substituteIn sama kuin sisään tulleen substituteOut samassa ottelussa. Jos olisi, jokaiselle vaihdolle löytyisi pari. Kysely laskee pois vaihdetut (substituteIn <> 0), sisään tulleet (substituteOut <> 0) ja yhdistyneet parit.

Tulos: pois vaihdettuja ja sisään tulleita on kumpiakin 77 559, eli jokaisella vaihdolla on yksi kumpaakin, mutta pareja löytyi vain 329. Tunniste ei siis ole sama vaihdon molemmilla puolilla.

In [0]:
spark.sql("""
    WITH off AS (
        SELECT gameID, substituteIn AS sub_id, playerID AS player_off,
               position AS pos_off, time AS minute_off
        FROM bronze_appearances
        WHERE substituteIn <> 0
    ),
    on_ AS (
        SELECT gameID, substituteOut AS sub_id, playerID AS player_on,
               position AS pos_on, time AS played_on
        FROM bronze_appearances
        WHERE substituteOut <> 0
    )
    SELECT
        (SELECT COUNT(*) FROM off) AS pois_vaihdettuja,
        (SELECT COUNT(*) FROM on_) AS sisaan_tulleita,
        COUNT(*) AS pareja
    FROM off JOIN on_ USING (gameID, sub_id)
""").show()

Ovatko löydetyt parit aitoja?

Kysely näyttää esimerkkejä 329 yhdistyneestä parista. minute_off on pois vaihdetun pelaajan lähtöminuutti ja played_on sisään tulleen vaihtopelaajan pelattu aika. Jos pari olisi aito, vaihtopelaaja tulee kentälle samalla minuutilla kun toinen lähtee, jolloin summa (minute_off + played_on) olisi noin 90.

Tulos: summat vaihtelevat välillä 48–82, joten mikään tarkastelluista pareista ei täsmää. Parit ovat sattumaa, koska saman ottelun tunnisteet ovat lähellä toisiaan.

Johtopäätös: substituteIn ja substituteOut ovat rivitunnisteita, joiden vastinsaraketta ei ole datassa, eikä niillä voi yhdistää vaihtoja pareiksi. Silver-tasolla ne korvataan kahdella selkeällä merkinnällä: pois vaihdettu (substituteIn <> 0) ja vaihtopelaajana tullut (substituteOut <> 0). Vaihtopelaajan tulominuutti lasketaan 90 − time.

In [0]:
spark.sql("""
    WITH off AS (
        SELECT gameID, substituteIn AS sub_id, playerID AS player_off,
               position AS pos_off, time AS minute_off
        FROM bronze_appearances
        WHERE substituteIn <> 0
    ),
    on_ AS (
        SELECT gameID, substituteOut AS sub_id, playerID AS player_on,
               position AS pos_on, time AS played_on
        FROM bronze_appearances
        WHERE substituteOut <> 0
    )
    SELECT gameID, sub_id, player_off, pos_off, minute_off,
           player_on, pos_on, played_on,
           minute_off + played_on AS summa
    FROM off JOIN on_ USING (gameID, sub_id)
    ORDER BY gameID, sub_id
""").show(15, truncate=False)